In [30]:
# Import libraries
from matminer.featurizers.composition import  Miedema, WenAlloys, ElementProperty, WenAlloys, YangSolidSolution
from matminer.featurizers.base import MultipleFeaturizer
from pymatgen.core.composition import Composition
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [31]:
input_csv_path = r"D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\raw\\hydride_prefeaturizing.csv"
output_csv_path = r"D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\processed\\preprocessed_features_matminer.csv"
df = pd.read_csv(input_csv_path)
df.head()

,Composition,Alloy_class,Dataset,temperature(K),Citation,log_H2_W% (max)
0,Mg1.5Ni1,Mg,HydPark,573,DOE_data_url,1.280934
1,Mg0.872Al0.128,Mg,HydPark,625,DOE_data_url,2.041220
2,Mg0.996Ni0.004,Mg,HydPark,573,DOE_data_url,1.722767
3,Mg0.807Al0.193,Mg,HydPark,625,DOE_data_url,2.028148
4,Mg0.879Ni0.121,Mg,HydPark,596,DOE_data_url,1.902108


In [35]:
# Define feature extraction function with error handling for invalid compositions


def generate_features_matminer(df, target_col='H2_W% (max)', formula_col='Composition', temperature_col='temperature(K)'):
    """
    Extracts features from a DataFrame using Matminer featurizers, focusing on Miedema for enthalpy of formation and mixing.
    
    Parameters:
    - df: DataFrame with formula, target, and temperature columns
    - target_col: Name of the target column (e.g., 'Wt%')
    - formula_col: Name of the formula column (e.g., 'formula')
    - temperature_col: Name of the temperature column (e.g., 'T')
    
    Returns:
    - X: DataFrame with extracted features
    - y: Series with target values
    - formulae: Series with formula values
    - skipped: List of tuples (index, formula) for skipped rows due to parsing errors
    - all_data: DataFrame with composition, T, Wt%, and all extracted features
    """
    df = df.copy()
    
    # Ensure numeric columns have proper dtypes before featurization
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Convert formulae to pymatgen Composition objects, handle parsing errors
    skipped = []
    def parse_composition(formula, index):
        try:
            if pd.notnull(formula):
                return Composition(formula)
            else:
                skipped.append((index, formula))
                return np.nan
        except ValueError as e:
            skipped.append((index, formula))
            return np.nan
    
    # Apply parsing with error handling
    df['composition'] = [parse_composition(formula, idx) for idx, formula in df[formula_col].items()]
    
    # Drop rows where composition parsing failed
    skipped.extend(df[df['composition'].isnull()].index.tolist())
    df = df.dropna(subset=['composition'])
    
    # Define Featurizers (focus on Miedema for enthalpy features)
    featurizers = [
        Miedema(impute_nan=False)
    ]
    
    # Combine featurizers into a MultipleFeaturizer
    multi_featurizer = MultipleFeaturizer(featurizers)
    
    # Extract features
    X = multi_featurizer.featurize_dataframe(df, col_id='composition', ignore_errors=True)
    
    # Add temperature as a feature
    X[temperature_col] = df[temperature_col]
    
    # Create all_data with formula, T, Wt%, and all features
    all_data = X.copy()
    
    
    # Handle NaN features
    feature_cols = [col for col in X.columns if col not in [formula_col, 'composition', target_col, temperature_col]]
    nan_compositions = X[X[feature_cols].isna().any(axis=1)].copy()
    X = X.dropna(subset=feature_cols)

    
    # Extract target and formulae
    y = X[target_col]
    formulae = X[formula_col]
    

    return X, y, formulae, skipped, all_data, nan_compositions

In [36]:
# Apply to training, validation, and test sets with correct unpacking
X, y, formulae, skipped, all_data, nan_compositions = generate_features_matminer(
    df, target_col='log_H2_W% (max)', formula_col='Composition', temperature_col='temperature(K)'
)

c:\Users\Asus\miniconda3\envs\Project\lib\site-packages\matminer\featurizers\composition\alloy.py:112: UserWarning: Miedema(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


MultipleFeaturizer:   0%|          | 0/1279 [00:00<?, ?it/s]

In [37]:
print("Saving preprocessed features to CSV...")
all_data.to_csv(output_csv_path, index=False)
print(f"Preprocessed features saved to {output_csv_path}")

Saving preprocessed features to CSV...
Preprocessed features saved to D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\processed\\preprocessed_features_matminer.csv


In [38]:
print(all_data.shape)
all_data.head()

(1279, 10)


,Composition,Alloy_class,Dataset,temperature(K),Citation,log_H2_W% (max),composition,Miedema_deltaH_inter,Miedema_deltaH_amor,Miedema_deltaH_ss_min
0,Mg1.5Ni1,Mg,HydPark,573,DOE_data_url,1.280934,"(Mg, Ni)",-0.074764,-0.021779,-0.037030
1,Mg0.872Al0.128,Mg,HydPark,625,DOE_data_url,2.041220,"(Mg, Al)",-0.083255,-0.047425,-0.064986
2,Mg0.996Ni0.004,Mg,HydPark,573,DOE_data_url,1.722767,"(Mg, Ni)",-0.000796,0.032766,0.001191
3,Mg0.807Al0.193,Mg,HydPark,625,DOE_data_url,2.028148,"(Mg, Al)",-0.126061,-0.085832,-0.092469
4,Mg0.879Ni0.121,Mg,HydPark,596,DOE_data_url,1.902108,"(Mg, Ni)",-0.023107,0.014270,0.023548


In [39]:
print(f"Feature extraction completed. Number of skipped compositions: {len(skipped)}")
if skipped:
    print(f"Skipped compositions: {skipped}")

Feature extraction completed. Number of skipped compositions: 0


In [ ]:
all_data.columns.to_list()

['Composition',
 'Alloy_class',
 'Dataset',
 'temperature(K)',
 'Citation',
 'log_H2_W% (max)',
 'composition',
 'Miedema_deltaH_inter',
 'Miedema_deltaH_amor',
 'Miedema_deltaH_ss_min']

: 